# Dentro de la foto · foto → escena 3D para Quest

Todo corre en una GPU gratis de Google. No necesitas PC.

1. Arriba: **Entorno de ejecución → Cambiar tipo de entorno → T4 GPU**.
2. **Entorno de ejecución → Ejecutar todo**. Te va a pedir permiso de Google Drive: acéptalo.
3. La primera vez se crea en tu Drive la carpeta **Dentro de la foto/fotos**. Sube ahí tus fotos (desde la app de Drive) y vuelve a darle **Ejecutar todo**.
4. Las escenas salen en **Dentro de la foto** (archivos `-quest.ply`). En el Quest: drive.google.com → descarga el `-quest.ply` → abre el visor (https://valibii.github.io/dentro-de-la-foto/) → **Cargar mi .ply** → **Entrar con passthrough**.

Solo convierte las fotos nuevas; las que ya tienen su `-quest.ply` se saltan.

In [ ]:
#@title 1 · Preparar (la primera vez tarda ~3-4 min)
import os, shutil, pathlib, torch
if not torch.cuda.is_available():
    raise SystemExit("⚠️ No hay GPU. Ve a Entorno de ejecución > Cambiar tipo de entorno > T4 GPU y vuelve a ejecutar.")
print("GPU:", torch.cuda.get_device_name(0))
if not os.path.exists("/content/ml-sharp/pyproject.toml"):
    !git clone -q --depth 1 https://github.com/apple/ml-sharp /content/ml-sharp
    !pip install -q -e /content/ml-sharp
!wget -q -nc -O /content/sharp.pt https://ml-site.cdn-apple.com/models/sharp/sharp_2572gikvuh.pt
from google.colab import drive
drive.mount("/content/drive")
DRIVE = pathlib.Path("/content/drive/MyDrive/Dentro de la foto")
FOTOS = DRIVE / "fotos"
FOTOS.mkdir(parents=True, exist_ok=True)
print("✅ Listo. Carpeta de fotos en Drive: Dentro de la foto/fotos")

In [ ]:
#@title 2 · Tomar las fotos nuevas de Drive
EXT = {".jpg", ".jpeg", ".png", ".heic", ".heif", ".webp"}
ENTRADA = pathlib.Path("/content/entrada")
shutil.rmtree(ENTRADA, ignore_errors=True); ENTRADA.mkdir()
nuevas = []
for f in sorted(FOTOS.iterdir()):
    if f.suffix.lower() not in EXT: continue
    stem = f.stem.replace(" ", "_")
    if (DRIVE / f"{stem}-quest.ply").exists(): continue
    shutil.copy(f, ENTRADA / (stem + f.suffix.lower())); nuevas.append(f.name)
if not nuevas:
    raise SystemExit("📷 No hay fotos nuevas. Sube tus fotos a Drive > 'Dentro de la foto' > 'fotos' y vuelve a darle Ejecutar todo.")
print("Fotos a convertir:", nuevas)

In [ ]:
#@title 3 · Convertir a 3D
MAX_SPLATS_QUEST = 700000 #@param {type:"integer"}
VIDEO_DE_PRUEBA = False #@param {type:"boolean"}
#@markdown `VIDEO_DE_PRUEBA` hace además un video con movimiento de cámara 3D (la primera vez compila y tarda varios minutos más).
import numpy as np
from plyfile import PlyData, PlyElement
SALIDA = pathlib.Path("/content/salida")
shutil.rmtree(SALIDA, ignore_errors=True); SALIDA.mkdir()
extra = "--render" if VIDEO_DE_PRUEBA else ""
!sharp predict -i /content/entrada -o /content/salida -c /content/sharp.pt {extra}

def optimizar(src, dst, max_splats):
    v = PlyData.read(str(src))["vertex"].data
    n0 = len(v)
    alpha = 1 / (1 + np.exp(-v["opacity"]))
    keep = alpha > 0.04
    v, alpha = v[keep], alpha[keep]
    if len(v) > max_splats:
        size = np.exp((v["scale_0"] + v["scale_1"] + v["scale_2"]) / 3)
        idx = np.argpartition(-(alpha * np.sqrt(size)), max_splats)[:max_splats]
        v = v[np.sort(idx)]
    PlyData([PlyElement.describe(v, "vertex")]).write(str(dst))
    print(f"{dst.name}: {n0:,} → {len(v):,} splats, {dst.stat().st_size/1e6:.0f} MB")

for ply in sorted(SALIDA.glob("*.ply")):
    optimizar(ply, ply.with_name(ply.stem + "-quest.ply"), MAX_SPLATS_QUEST)

In [ ]:
#@title 4 · Guardar en Google Drive
for f in sorted(SALIDA.glob("*-quest.ply")) + sorted(SALIDA.glob("*.mp4")):
    shutil.copy(f, DRIVE / f.name); print("Drive ←", f.name)
print("✅ Listo. En el Quest: drive.google.com → 'Dentro de la foto' → descarga el -quest.ply → ábrelo en el visor.")